

Ishlatish: `DATASET_KEY` qiymatini tanlang, dastlab `MAX_DOCS = 1000` qoldiring va kataklarni yuqoridan pastga bajaring Yuklashdan keyin barcha yozuvlar `{"text": "..."}` ko‘rinishida bo‘ladi; `dataset[0]`, `dataset[2]` va `len(dataset)` ishlaydi.



bu manba inglizcha. SimpleStories qisqa sintetik hikoyalardan iborat; hikoya matnlarini beradi.

Manba: [SimpleStories](https://huggingface.co/datasets/SimpleStories/SimpleStories)



Uchala manba `load_dataset(..., streaming=True)` orqali yuklandi.


Qo‘shimcha tekshiruv: yangi hujjat tanlash qismi 1 0000 ta haqiqiy SimpleStories yozuvida bajarildi — 10 ta o‘tgan misol va rad etilganlar 0 ekanligi chiqarildi. Rad sabablarining to‘rttalasi, bir hujjatning bir nechta shartni buzishi, 10 tadan kam/0 ta misol va 10+10 tanlash chegarasi alohida sun’iy sinov yozuvlarida tekshirildi. Sun’iy yozuvlar korpusga qo‘shilmadi.


In [5]:
from pathlib import Path
import re
import hashlib
import json
import unicodedata
import html
import sys
import importlib.util
from itertools import islice
from collections import Counter

In [ ]:
# Faqat shu qiymatni almashtirib, boshqa datasetni tanlaymiz:
# "simplestories", "children" yoki "cosmopedia_stories".
DATASET_KEY = "simplestories"

# Dastlab faqat 1 0000 ta yozuvni tekshiramiz.
# Keyin kompyuter xotirasiga mos ravishda oshirishimiz mumkin.
MAX_DOCS = 10000

DATASET_OPTIONS = {
    "simplestories": {
        "path": "SimpleStories/SimpleStories",
        "config": None,
        "text_column": "story",
        "split": "train",
    },
    "children": {
        "path": "ajibawa-2023/Children-Stories-Collection",
        "config": None,
        "text_column": "text",
        "split": "train",
    },
    "cosmopedia_stories": {
        "path": "HuggingFaceTB/cosmopedia",
        "config": "stories",
        "text_column": "text",
        "split": "train",
    },
}

if DATASET_KEY not in DATASET_OPTIONS:
    raise ValueError(f"DATASET_KEY quyidagilardan biri bo'lsin: {list(DATASET_OPTIONS)}")
if not isinstance(MAX_DOCS, int) or isinstance(MAX_DOCS, bool) or MAX_DOCS <= 0:
    raise ValueError("MAX_DOCS musbat butun son bo'lishi kerak.")

DATASET_CONFIG = DATASET_OPTIONS[DATASET_KEY]

# Natijalar notebook ishga tushirilgan papka ichiga yoziladi.
# Kerak bo'lsa, bu yerda loyihangizning aniq yo'lini belgilaymiz.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / DATASET_KEY
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
FILTERED_DIR = DATA_DIR / "filtered"
FINAL_DIR = DATA_DIR / "final"

for directory in [RAW_DIR, CLEANED_DIR, FILTERED_DIR, FINAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Tanlangan dataset:", DATASET_CONFIG["path"])
print("Manbadagi matn ustuni:", DATASET_CONFIG["text_column"])
print("Ko'riladigan yozuvlar chegarasi:", MAX_DOCS)
print("Natijalar papkasi:", DATA_DIR.resolve())

Tanlangan dataset: SimpleStories/SimpleStories
Manbadagi matn ustuni: story
Ko'riladigan yozuvlar chegarasi: 10000
Natijalar papkasi: C:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\4 dars\data\simplestories


In [7]:
# Jupyter ishlatayotgan Python muhitiga faqat zarur bo'lsa o'rnatiladi.
# Har safar pyarrow/datasets paketlarini o'chirib qayta o'rnatmang.
if importlib.util.find_spec("datasets") is None:
    get_ipython().run_line_magic("pip", "install -q datasets")

In [8]:
import datasets
import pyarrow

print("Python:", sys.version.split()[0])
print("Python muhiti:", sys.executable)
print("datasets:", datasets.__version__)
print("pyarrow:", pyarrow.__version__)

Python: 3.11.9
Python muhiti: C:\Users\xolmu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
datasets: 5.1.0
pyarrow: 26.0.0


In [9]:
from datasets import Dataset, load_dataset

# streaming=True: butun korpusni oldindan yuklab olmaymiz.
# Tarmoq va fayl formati sabab kichik namunaga ham bir qancha MB o'qilishi mumkin.
source = load_dataset(
    DATASET_CONFIG["path"],
    name=DATASET_CONFIG["config"],
    split=DATASET_CONFIG["split"],
    streaming=True,
)

source_text_column = DATASET_CONFIG["text_column"]
records = []
source_columns = None
loading_stats = {"seen": 0, "non_string_removed": 0, "empty_removed": 0, "output": 0}

# Birinchi MAX_DOCS yozuv olinadi: bu tezkor sinov, tasodifiy tanlama emas.
for item in islice(source, MAX_DOCS):
    loading_stats["seen"] += 1
    if source_columns is None:
        source_columns = list(item.keys())
        print("Manbaning ustunlari:", source_columns)

    if source_text_column not in item:
        raise KeyError(
            f"'{source_text_column}' ustuni topilmadi. "
            f"Mavjud ustunlar: {list(item.keys())}"
        )

    value = item[source_text_column]
    if not isinstance(value, str):
        loading_stats["non_string_removed"] += 1
        continue
    if not value.strip():
        loading_stats["empty_removed"] += 1
        continue

    # Barcha manbalar keyingi kataklar kutadigan yagona formatga o'tadi.
    records.append({"text": value})

if not records:
    raise ValueError("Tanlangan namunada bo'sh bo'lmagan matn topilmadi.")

# Oddiy Dataset qaytaramiz: dataset[0], len(dataset), column_names ishlaydi.
# Xotiraga faqat chegaralangan namuna joylanadi.
dataset = Dataset.from_list(records)
loading_stats["output"] = len(dataset)
del records

print("Yuklash statistikasi:", loading_stats)
print(dataset)

Manbaning ustunlari: ['story', 'topic', 'theme', 'style', 'feature', 'grammar', 'persona', 'initial_word_type', 'initial_letter', 'word_count', 'character_count', 'num_paragraphs', 'avg_word_length', 'avg_sentence_length', 'flesch_reading_ease', 'flesch_kincaid_grade', 'dale_chall_readability_score', 'num_stories_in_completion', 'expected_num_stories_in_completion', 'generation_id', 'model']
Yuklash statistikasi: {'seen': 10000, 'non_string_removed': 0, 'empty_removed': 0, 'output': 10000}
Dataset({
    features: ['text'],
    num_rows: 10000
})


In [10]:
print("Moslashtirilgan ustunlar:", dataset.column_names)
print("Birinchi matn:", dataset[0]["text"][:500])

Moslashtirilgan ustunlar: ['text']
Birinchi matn: Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. 

Inspired, the children played with the animals and shared their own stories of hardship. They realized that laughter could help them through hard times. When they finally left, they took the animals' smiles with 


In [11]:
index = min(2, len(dataset) - 1)
print(f"{index + 1}-yozuv:", dataset[index]["text"][:500])

3-yozuv: Rain poured down on the tiny town, and I watched from my secret lair. I, a master of evil, had an idea brewing in my mind. "Today, I will flood the town!" I chuckled wickedly. My friend, Samuel, was completely oblivious. He stood in the rain, trying to make paper boats. "Look! They float!" he yelled, splashing around.

"Float? They will sink!" I said with a smirk. Samuel just smiled wider. "Then I'll make more! This is so much fun!" I couldn't believe it. How could he find joy in something so si


In [12]:
print("Namuna hajmi:", len(dataset))

Namuna hajmi: 10000


In [13]:
raw_file = RAW_DIR / f"{DATASET_KEY}_train.jsonl"

with open(raw_file, "w", encoding="utf-8") as f:
    for item in dataset:
        record = {
            "text": item["text"]
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved raw data to: {raw_file}")
print(f"File size: {raw_file.stat().st_size / (1024**2):.2f} MB")

Saved raw data to: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\4 dars\data\simplestories\raw\simplestories_train.jsonl
File size: 12.05 MB


**Boshlang‘ich matnlar statistikasi.** JSONL faylning har bir qatorida bitta `{"text": "..."}` yozuvi bo‘ladi.

In [14]:
def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    return records


raw_docs = load_jsonl(raw_file)

print("Number of raw documents:", len(raw_docs))

Number of raw documents: 10000


In [15]:
raw_chars = sum(len(doc["text"]) for doc in raw_docs)

print("Raw documents:", len(raw_docs))
print("Raw characters:", raw_chars)
print("Raw size MB:", round(raw_file.stat().st_size / (1024**2), 2))

Raw documents: 10000
Raw characters: 12356247
Raw size MB: 12.05


**Matnni tozalash.** Quyida HTML belgilari, Unicode yozilishi, boshqaruv belgilari va ortiqcha bo‘shliqlar tartibga keltiriladi.

Boshlang‘ich yozuvlar tozalash funksiyasidan o‘tib, tozalangan yozuvlar ro‘yxatiga yig‘iladi.

HTML entity, masalan `&amp;`, odatiy belgiga aylantiriladi. HTML teglar olib tashlanadi. Unicode NFKC normallashtirish qo‘llanadi. Bo‘sh matnlar chiqariladi.

Bu funksiya oddiy hikoya matnlari uchun moslangan. Kod va matematik yozuvli korpuslarda HTML tozalash qoidalarini alohida baholang.

In [16]:
# class DataCleaner:

def remove_html(text):
    """
    Remove HTML/XML-like tags.
    """
    text = re.sub(r"<[^>]+>", " ", text)    
    return text


def normalize_unicode(text):
    """
    Normalize Unicode using NFKC.
    """
    return unicodedata.normalize("NFKC", text)


def remove_control_characters(text):
    """
    Remove control characters while keeping
    newline and tab.
    """
    return "".join(
        char for char in text
        if char in "\n\t" or not unicodedata.category(char).startswith("C")
    )


def normalize_whitespace(text):
    """
    Compress repeated whitespace.
    """
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def clean_text(text):
    """
    Complete cleaning pipeline.
    """
    text = html.unescape(text)
    text = remove_html(text)
    text = normalize_unicode(text)
    text = remove_control_characters(text)
    text = normalize_whitespace(text)

    return text

In [17]:
#Statitisktika uchun
cleaned_docs = []

cleaning_stats = {
    "input": 0,
    "output": 0,
    "empty_removed": 0
}

for doc in raw_docs:

    cleaning_stats["input"] += 1

    original_text = doc["text"]
    cleaned_text = clean_text(original_text)

    if not cleaned_text:
        cleaning_stats["empty_removed"] += 1
        continue

    cleaned_docs.append({
        "text": cleaned_text
    })

cleaning_stats["output"] = len(cleaned_docs)

print(cleaning_stats)

{'input': 10000, 'output': 10000, 'empty_removed': 0}


In [18]:
#tozalangan datani saqlash
cleaned_file = CLEANED_DIR / f"{DATASET_KEY}_cleaned.jsonl"

with open(cleaned_file, "w", encoding="utf-8") as f:
    for doc in cleaned_docs:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )

print(f"Saved cleaned data to: {cleaned_file}")

Saved cleaned data to: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\4 dars\data\simplestories\cleaned\simplestories_cleaned.jsonl


**Matnlarni filtrlash.** Har bir matn quyidagi chegaralar bilan tekshiriladi.

`min_chars` va `max_chars` belgilar sonini, `min_alpha_ratio` harflar ulushini, `max_repeated_line_ratio` takroriy qatorlar ulushini belgilaydi. 200 belgidan qisqa hikoya o‘chirilsa, bu dastur xatosi emas — filtr natijasi.

In [19]:
FILTER_CONFIG = {
    "min_chars": 200,
    "max_chars": 100000,
    "min_alpha_ratio": 0.30,
    "max_repeated_line_ratio": 0.30
}

FILTER_CONFIG

{'min_chars': 200,
 'max_chars': 100000,
 'min_alpha_ratio': 0.3,
 'max_repeated_line_ratio': 0.3}

In [20]:
def alphabetic_ratio(text):
    """
    Percentage of characters that are alphabetic.
    """
    if not text:
        return 0.0

    alpha_count = sum(char.isalpha() for char in text)

    return alpha_count / len(text)


def repeated_line_ratio(text):
    """
    Ratio of repeated non-empty lines.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) <= 1:
        return 0.0

    counts = Counter(lines)

    repeated_lines = sum(
        count
        for count in counts.values()
        if count > 1
    )

    return repeated_lines / len(lines)

In [21]:
# Har bir hujjatning qarori, barcha rad sabablari va ko'rsatkichlarini saqlaymiz.
filtered_docs = []
rejected_docs = []
filter_audit = []

# Keyin sozlamani o'zgartirsangiz ham, bu tekshiruv chegaralari saqlanadi.
filter_config_used = dict(FILTER_CONFIG)

filter_stats = {
    "input": len(cleaned_docs),
    "min_chars": 0,
    "max_chars": 0,
    "alpha_ratio": 0,
    "repeated_lines": 0,
    "output": 0,
}

for cleaned_index, doc in enumerate(cleaned_docs):
    text = doc["text"]
    metrics = {
        "chars": len(text),
        "alpha_ratio": alphabetic_ratio(text),
        "repeated_line_ratio": repeated_line_ratio(text),
    }

    # Bitta hujjat bir nechta shartni buzishi mumkin: barchasini qayd etamiz.
    failed_rules = []
    if metrics["chars"] < filter_config_used["min_chars"]:
        failed_rules.append("min_chars")
    if metrics["chars"] > filter_config_used["max_chars"]:
        failed_rules.append("max_chars")
    if metrics["alpha_ratio"] < filter_config_used["min_alpha_ratio"]:
        failed_rules.append("alpha_ratio")
    if metrics["repeated_line_ratio"] > filter_config_used["max_repeated_line_ratio"]:
        failed_rules.append("repeated_lines")

    audit_record = {
        "cleaned_index": cleaned_index,
        "status": "rejected" if failed_rules else "passed",
        "text": text,
        "metrics": metrics,
        "failed_rules": failed_rules,
        "first_failed_rule": failed_rules[0] if failed_rules else None,
        "limits": filter_config_used,
    }
    filter_audit.append(audit_record)

    if failed_rules:
        # Asl statistikadagi tartib saqlanadi: hujjat birinchi sababda sanaladi.
        filter_stats[failed_rules[0]] += 1
        rejected_docs.append(audit_record)
        continue

    # Keyingi bosqichlar uchun format avvalgidek qoladi.
    filtered_docs.append(doc)

filter_stats["output"] = len(filtered_docs)

print("Filtr statistikasi:", filter_stats)
print("Filtrdan o'tgan hujjatlar:", len(filtered_docs))
print("Rad etilgan hujjatlar:", len(rejected_docs))
print("Tekshiruv yozuvlari:", len(filter_audit))

Filtr statistikasi: {'input': 10000, 'min_chars': 0, 'max_chars': 0, 'alpha_ratio': 0, 'repeated_lines': 0, 'output': 10000}
Filtrdan o'tgan hujjatlar: 10000
Rad etilgan hujjatlar: 0
Tekshiruv yozuvlari: 10000


Quyidagi tekshiruv **filtrlash bosqichi** uchun: 10 ta o‘tgan va 10 ta rad etilgan hujjatni sababi bilan o‘qib chiqasiz. Takrorlarni olib tashlash va PII niqoblash keyingi bosqichlarda bajariladi.

`filter_audit` — har bir hujjat bo‘yicha qaror va ko‘rsatkichlar ro‘yxati. `rejected_docs` — rad etilgan hujjatlar, ularning matni va sabablari. `filtered_docs` — keyingi bosqichlarga uzatiladigan matnlar. `cleaned_index` — `cleaned_docs` ichidagi, 0 dan boshlanadigan indeks.

Bitta hujjat bir nechta filtr shartini buzsa, barcha sabablar ko‘rsatiladi. Umumiy `filter_stats`da esa asl koddagidek faqat birinchi sabab sanaladi: shu sababli hujjat ikki marta sanalmaydi.

Har bir guruhdan tasodifiy 10 ta yozuv olinadi. `seed=42` tanlovni takrorlash uchun ishlatiladi. Guruhda 10 tadan kam hujjat bo‘lsa, borlari ko‘rsatiladi; 0 ta bo‘lsa, bu holat ochiq yoziladi. Avval tekshirilgan 1 000 ta SimpleStories yozuvida rad etilganlar soni 0 edi. Ko‘proq namuna olish ham rad etilgan hujjat chiqishini kafolatlamaydi.

In [22]:
import random


def show_filter_examples(audit_records, status, n=10, seed=42):
    """Tanlangan guruhdan ko'pi bilan n ta matnni to'liq chop etadi."""
    if status not in {"passed", "rejected"}:
        raise ValueError("status 'passed' yoki 'rejected' bo'lishi kerak.")
    if not isinstance(n, int) or isinstance(n, bool) or n < 0:
        raise ValueError("n manfiy bo'lmagan butun son bo'lishi kerak.")

    candidates = [r for r in audit_records if r["status"] == status]
    selected = random.Random(seed).sample(candidates, min(n, len(candidates)))
    group_name = "FILTRDAN O'TGAN" if status == "passed" else "RAD ETILGAN"

    print(f"\n{group_name}: jami {len(candidates)} ta; ko'rsatilmoqda {len(selected)} ta.")
    if not candidates:
        print("Bu guruhda hujjat yo'q. Mavjud bo'lmagan misollar qo'shilmaydi.")
        if status == "rejected":
            print("Ushbu namunadagi barcha hujjatlar belgilangan filtr shartlaridan o'tgan.")
        return []

    for sample_number, record in enumerate(selected, start=1):
        metrics = record["metrics"]
        limits = record["limits"]
        print("\n" + "=" * 80)
        print(f"{sample_number}-MISOL | cleaned_docs indeksi: {record['cleaned_index']}")
        print(f"Qaror: {group_name}")
        print(
            f"Belgilar soni: {metrics['chars']} "
            f"(ruxsat: {limits['min_chars']} dan {limits['max_chars']} gacha)"
        )
        print(
            f"Harflar ulushi: {metrics['alpha_ratio']:.3f} "
            f"(kamida {limits['min_alpha_ratio']:.3f})"
        )
        print(
            f"Takroriy qatorlar ulushi: {metrics['repeated_line_ratio']:.3f} "
            f"(ko'pi bilan {limits['max_repeated_line_ratio']:.3f})"
        )

        explanations = {
            "min_chars": f"Matn qisqa: {metrics['chars']} < {limits['min_chars']} belgi",
            "max_chars": f"Matn uzun: {metrics['chars']} > {limits['max_chars']} belgi",
            "alpha_ratio": (
                f"Harflar kam: {metrics['alpha_ratio']:.3f} < "
                f"{limits['min_alpha_ratio']:.3f}"
            ),
            "repeated_lines": (
                f"Takroriy qatorlar ko'p: {metrics['repeated_line_ratio']:.3f} > "
                f"{limits['max_repeated_line_ratio']:.3f}"
            ),
        }
        if record["failed_rules"]:
            print("Birinchi rad sababi:", record["first_failed_rule"])
            print("Buzilgan barcha shartlar:")
            for reason in record["failed_rules"]:
                print(f"  - {reason}: {explanations[reason]}")
        else:
            print("Rad sababi: yo'q; barcha to'rtta shart bajarilgan.")

        # Matnni [:500] bilan kesmaymiz: to'liq o'qish mumkin.
        print("\nTO'LIQ MATN:")
        print(record["text"])

    return selected


# seed bir xil bo'lsa, bir xil tartibdagi yozuvlardan bir xil namunalar tanlanadi.
passed_examples = show_filter_examples(filter_audit, status="passed", n=10, seed=42)
rejected_examples = show_filter_examples(filter_audit, status="rejected", n=10, seed=42)


FILTRDAN O'TGAN: jami 10000 ta; ko'rsatilmoqda 10 ta.

1-MISOL | cleaned_docs indeksi: 1824
Qaror: FILTRDAN O'TGAN
Belgilar soni: 696 (ruxsat: 200 dan 100000 gacha)
Harflar ulushi: 0.780 (kamida 0.300)
Takroriy qatorlar ulushi: 0.000 (ko'pi bilan 0.300)
Rad sababi: yo'q; barcha to'rtta shart bajarilgan.

TO'LIQ MATN:
Swiftly, Kim runs into the old theater, its doors wide open. She can hear the echoes of laughter and music, but the seats are empty. She feels a chill run down her spine, but her love for the stage drives her on. As she steps onto the stage, the lights flicker on, revealing a ghostly audience. They watch her with eager eyes. 

Kim takes a deep breath and begins to sing. Her voice fills the room, and the ghosts sway gently in their seats. With every note, she feels their energy surround her, lifting her spirit. After her song, a spotlight shines on a single empty seat. It belongs to the greatest performer of all time, and she knows she has made a connection. She will keep 

**Qaysi filtr xato ishlaganini matnni o‘qib baholaysiz.** Kod qaysi shart buzilganini aniq ko‘rsatadi; shartning maqsadga to‘g‘ri kelishini inson baholaydi.

| Kuzatilgan holat | Talqin |
|---|---|
| Sifatli, maqsadga mos matn filtrdan o‘tdi | Shu misolda qabul qarori mos |
| Sifatsiz yoki maqsadga mos bo‘lmagan matn rad etildi | Shu misolda rad qarori mos |
| Sifatli, kerakli matn rad etildi | Ortiqcha filtrlash bo‘lishi mumkin; sabab va chegarani tekshiring |
| Sifatsiz yoki maqsadga mos bo‘lmagan matn o‘tdi | Mavjud filtrlar bu kamchilikni tekshirmayotgan yoki chegarasi yetarli bo‘lmagan bo‘lishi mumkin |

Masalan, 150 belgili mazmunli hikoya `min_chars=200` sababli rad etilsa, bu loyiha uchun uzunlik chegarasi ortiqcha qattiq bo‘lishi mumkin. Lekin talabingizning o‘zi kamida 200 belgi bo‘lsa, kod belgilangan talabni bajargan bo‘ladi. Keraksiz ravishda chegarani o‘zgartirib, sun’iy ravishda 10 ta rad etilgan misol yaratmang.

`min_alpha_ratio=0.30` — barcha belgilar ichida harflar kamida 30% bo‘lishini bildiradi. `max_repeated_line_ratio=0.30` uchun hozirgi funksiya takrorlangan qatorlarning **barcha nusxalarini** sanaydi. Masalan, uch qatorning ikkitasi aynan bir xil bo‘lsa, ulush 2/3 bo‘ladi. She’r yoki qaytarma jumlali hikoyada bu alohida baholanishi kerak.

Bu to‘rtta filtr matnning faktlari, mazmuni, grammatikasi yoki tilini alohida tekshirmaydi. Shuning uchun “o‘tdi” degani “barcha jihatdan sifatli” degani emas. 10+10 misol tezkor tekshiruv; butun korpus sifatini o‘lchash uchun kattaroq, vakil tanlama va qo‘lda belgilangan baholar kerak.

Ko‘rsatilgan yozuvga qaytish uchun, masalan, `cleaned_docs[17]["text"]` deb uning indeksidan foydalanishingiz mumkin. O‘zingiz o‘qigan haqiqiy indeksni qo‘ying.

In [23]:
#fiter bolgan datani saqlash
filtered_file = FILTERED_DIR / f"{DATASET_KEY}_filtered.jsonl"

with open(filtered_file, "w", encoding="utf-8") as f:
    for doc in filtered_docs:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )

print(f"Saved filtered data to: {filtered_file}")

Saved filtered data to: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\4 dars\data\simplestories\filtered\simplestories_filtered.jsonl


**Takrorlarni olib tashlash.** Bu bo‘lim normallashtirilgan matni aynan bir xil hujjatlarni ajratadi.

Chetdagi bo‘shliqlar olinadi, matn kichik harfga o‘tkaziladi va SHA-256 xeshi hisoblanadi. Shu sababli faqat harf kattaligi bilan farqlanuvchi matnlar ham bir guruhga kiradi. Mazmunan o‘xshash, lekin boshqa so‘zlar bilan yozilgan matnlar bu usulda topilmaydi.

In [24]:
def document_hash(text):
    """
    Create SHA-256 hash for a document.
    """
    normalized = text.strip().lower()

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()

In [25]:
unique_docs = []
seen_hashes = set()

duplicate_count = 0

for doc in filtered_docs:

    text = doc["text"]

    doc_hash = document_hash(text)

    if doc_hash in seen_hashes:
        duplicate_count += 1
        continue

    seen_hashes.add(doc_hash)
    unique_docs.append(doc)

print("Input documents:", len(filtered_docs))
print("Duplicates removed:", duplicate_count)
print("Unique documents:", len(unique_docs))

Input documents: 10000
Duplicates removed: 0
Unique documents: 10000


**Shaxsiy ma’lumotga o‘xshash belgilarni niqoblash.** PII — shaxsni aniqlashga yordam beruvchi ma’lumot.

Takrorlardan tozalangan matnlarga email, telefon va uzun raqam andozalari qo‘llanadi. Bu katak Markdown: Python kodi sifatida bajarilmaydi.

Quyidagi muntazam ifodalar email manzillari va telefon/uzun raqamga o‘xshash qismlarni qidiradi. Ular barcha shaxsiy ma’lumotlarni topishni kafolatlamaydi; oddiy sonlarni ham niqoblashi mumkin.

In [26]:
EMAIL_PATTERN = re.compile(
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)"
)

NUMBER_PATTERN = re.compile(
    r"(?<!\d)\d{6,}(?!\d)"
)

In [27]:
def scrub_pii(text):
    """
    Replace common PII patterns with placeholders.
    """

    stats = {
        "email": 0,
        "phone": 0,
        "digits": 0
    }

    text, stats["email"] = EMAIL_PATTERN.subn(
        "<EMAIL>",
        text
    )

    text, stats["phone"] = PHONE_PATTERN.subn(
        "<PHONE>",
        text
    )

    text, stats["digits"] = NUMBER_PATTERN.subn(
        "<NUMBER>",
        text
    )

    return text, stats

In [28]:
final_docs = []

pii_stats = {
    "email": 0,
    "phone": 0,
    "digits": 0
}

for doc in unique_docs:

    cleaned_text, stats = scrub_pii(
        doc["text"]
    )

    for key in pii_stats:
        pii_stats[key] += stats[key]

    final_docs.append({
        "text": cleaned_text
    })

print("PII matches:")
print(pii_stats)

PII matches:
{'email': 0, 'phone': 2, 'digits': 0}


In [29]:
#save final corpus
final_file = FINAL_DIR / f"{DATASET_KEY}_final.jsonl"

with open(final_file, "w", encoding="utf-8") as f:
    for doc in final_docs:
        f.write(
            json.dumps(doc, ensure_ascii=False) + "\n"
        )

print(f"Final corpus saved to: {final_file}")

Final corpus saved to: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\4 dars\data\simplestories\final\simplestories_final.jsonl


Natijalar `data/<DATASET_KEY>/` ichida saqlanadi:

| Papka | Natija |
|---|---|
| `raw` | Moslashtirilgan boshlang‘ich matnlar |
| `cleaned` | Tozalangan matnlar |
| `filtered` | Filtrdan o‘tgan matnlar |
| `final` | Yakuniy JSONL, statistik hisobot va manba sozlamalari |

Bir datasetni qayta ishga tushirish o‘sha datasetning natija fayllarini yangilaydi.

In [30]:
#final statistics( how many in how many out)
final_chars = sum(
    len(doc["text"])
    for doc in final_docs
)

stats_report = f"""# Data Pipeline Statistics

- Dataset: {DATASET_CONFIG["path"]}
- Config: {DATASET_CONFIG["config"]}
- Split: {DATASET_CONFIG["split"]}
- Source text column: {source_text_column}
- Input row limit: {MAX_DOCS}
- Loading statistics: {loading_stats}

## Raw

- Documents: {len(raw_docs)}
- Characters: {raw_chars}
- Size: {raw_file.stat().st_size / (1024**2):.2f} MB

## Cleaning

- Input documents: {cleaning_stats["input"]}
- Output documents: {cleaning_stats["output"]}
- Empty/broken documents removed: {cleaning_stats["empty_removed"]}

## Filtering

- Input documents: {filter_stats["input"]}
- Removed by min_chars: {filter_stats["min_chars"]}
- Removed by max_chars: {filter_stats["max_chars"]}
- Removed by alphabetic ratio: {filter_stats["alpha_ratio"]}
- Removed by repeated-line ratio: {filter_stats["repeated_lines"]}
- Output documents: {filter_stats["output"]}

## Deduplication

- Input documents: {len(filtered_docs)}
- Exact duplicates removed: {duplicate_count}
- Unique documents: {len(unique_docs)}

## PII Scrubbing

- Emails masked: {pii_stats["email"]}
- Phone numbers masked: {pii_stats["phone"]}
- Large digit sequences masked: {pii_stats["digits"]}

## Final

- Documents: {len(final_docs)}
- Characters: {final_chars}

## Pipeline

Raw
→ Cleaning
→ Filtering
→ Deduplication
→ PII Scrubbing
→ Final Corpus
"""

report_file = FINAL_DIR / "stats_report.md"

with open(report_file, "w", encoding="utf-8") as f:
    f.write(stats_report)

print(stats_report)
print(f"Report saved to: {report_file}")

# Data Pipeline Statistics

- Dataset: SimpleStories/SimpleStories
- Config: None
- Split: train
- Source text column: story
- Input row limit: 10000
- Loading statistics: {'seen': 10000, 'non_string_removed': 0, 'empty_removed': 0, 'output': 10000}

## Raw

- Documents: 10000
- Characters: 12356247
- Size: 12.05 MB

## Cleaning

- Input documents: 10000
- Output documents: 10000
- Empty/broken documents removed: 0

## Filtering

- Input documents: 10000
- Removed by min_chars: 0
- Removed by max_chars: 0
- Removed by alphabetic ratio: 0
- Removed by repeated-line ratio: 0
- Output documents: 10000

## Deduplication

- Input documents: 10000
- Exact duplicates removed: 0
- Unique documents: 10000

## PII Scrubbing

- Emails masked: 0
- Phone numbers masked: 2
- Large digit sequences masked: 0

## Final

- Documents: 10000
- Characters: 12356186

## Pipeline

Raw
→ Cleaning
→ Filtering
→ Deduplication
→ PII Scrubbing
→ Final Corpus

Report saved to: c:\Users\xolmu\OneDrive\Desktop\Darsl

In [31]:
stage_counts = {
    "Raw": len(raw_docs),
    "Cleaned": len(cleaned_docs),
    "Filtered": len(filtered_docs),
    "Deduplicated": len(unique_docs),
    "Final": len(final_docs)
}

for stage, count in stage_counts.items():
    retention = count / len(raw_docs) * 100

    print(
        f"{stage:15} : "
        f"{count:,} documents "
        f"({retention:.2f}% retained)"
    )

Raw             : 10,000 documents (100.00% retained)
Cleaned         : 10,000 documents (100.00% retained)
Filtered        : 10,000 documents (100.00% retained)
Deduplicated    : 10,000 documents (100.00% retained)
Final           : 10,000 documents (100.00% retained)


In [32]:
# Qaysi sozlamalar bilan olinganini qayta ko'rish uchun saqlaymiz.
manifest = {
    "dataset_key": DATASET_KEY,
    "source": DATASET_CONFIG,
    "source_columns": source_columns,
    "max_docs": MAX_DOCS,
    "sampling": "first_n_rows",
    "loading_stats": loading_stats,
    "filter_config": FILTER_CONFIG,
    "stage_counts": stage_counts,
    "versions": {
        "python": sys.version.split()[0],
        "datasets": datasets.__version__,
        "pyarrow": pyarrow.__version__,
    },
}
manifest_file = FINAL_DIR / "run_config.json"
manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Sozlamalar fayli:", manifest_file)

Sozlamalar fayli: c:\Users\xolmu\OneDrive\Desktop\Darslar oyma oy\LLM 2\4 dars\data\simplestories\final\run_config.json


In [33]:
for i, doc in enumerate(raw_docs[:5], start=1):
    print("=" * 60)
    print(f"{i}-HUJJAT: TOZALASHDAN OLDIN")
    print(doc["text"][:350])
    print("TOZALASHDAN KEYIN")
    print(clean_text(doc["text"])[:350])

1-HUJJAT: TOZALASHDAN OLDIN
Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. 

Inspired, the children played with the animals and shared thei
TOZALASHDAN KEYIN
Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. 

Inspired, the children played with the animals and shared thei
2-HUJJAT: TOZALASHDAN OLDIN
Key turned in the old lock. A girl named Lily held her breath. She had always wanted to open the door in her attic. It was covered in dust and shadows. It felt like a secret place, a mystery waiting to be solved. With one tw

In [34]:
for i, doc in enumerate(final_docs[:3], start=1):
    print("=" * 60)
    print(f"YAKUNIY HUJJAT {i}")
    print(doc["text"][:350])

YAKUNIY HUJJAT 1
Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. 

Inspired, the children played with the animals and shared thei
YAKUNIY HUJJAT 2
Key turned in the old lock. A girl named Lily held her breath. She had always wanted to open the door in her attic. It was covered in dust and shadows. It felt like a secret place, a mystery waiting to be solved. With one twist, the door creaked open, revealing a room full of toys. 

Lily saw a dollhouse that looked perfect. She ran to it and picke
YAKUNIY HUJJAT 3
Rain poured down on the tiny town, and I watched from my secret lair. I, a master of evil, had an idea brewing in my mind. "Today, I will flood the town!" I chuckled wickedly. My friend, Samuel, was completely oblivious. He stood in the rain, tryi

**Xatoni ajratish.** `KeyError: text` — noto‘g‘ri ustun; `BuilderConfig` xatosi — noto‘g‘ri dataset bo‘limi; `ModuleNotFoundError` — tanlangan Python muhitida paket yo‘q; `ConnectionError`/timeout — ulanish muammosi. Filtrdan keyin 0 ta hujjat qolsa, avval statistikadagi chiqarib tashlash sabablarini tekshiring.


Bu notebook tayyor matn korpusini yaratadi. Tokenlash va modelni o‘qitish keyingi bosqichdir.

Keng miqyosli training uchun millionlab hujjatni bir vaqtning o‘zida bir necha Python ro‘yxatida saqlash o‘rniga bo‘laklab qayta ishlash kerak. Hozirgi kichik namuna o‘quv tajribasi uchun mo‘ljallangan. Haqiqiy baholashda train va validation/test ma’lumotlari ajratiladi va ular orasidagi takrorlar nazorat qilinadi.